<a href="https://colab.research.google.com/github/williamli218/mais202-team-2-fraud-bursts/blob/main/notebooks/workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1.1 Setup

Run the code below in order to downlaod your repo onto Colab (using your GitHub token). It will then update it with git pull, load your Kaggle keys, install the needed packages, and make src/ and config.py importable. Run it once at the start of every session.

In [ ]:
REPO_URL = "https://github.com/williamli218/mais202-team-2-fraud-bursts"

import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if "<your-team>" in REPO_URL:
        raise ValueError("Edit REPO_URL at the top of this cell to your team's repository first.")
    from google.colab import userdata

    def _secret(name):
        try:
            return userdata.get(name)
        except Exception as e:
            raise RuntimeError(
                f"Colab Secret {name} is missing or not shared with this notebook. "
                "Open the key icon in the left sidebar, add it, and switch on Notebook access."
            ) from e

    token = _secret("GITHUB_TOKEN")
    for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        os.environ[name] = _secret(name)

    repo = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.isdir(repo):
        auth_url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--quiet", auth_url], check=True)
    os.chdir(repo)
    subprocess.run(["git", "pull", "--quiet"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # Not in Colab: assume this notebook is open from inside the repository.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
import config
print("repository:", os.getcwd(), "| template version:", config.TEMPLATE_VERSION)
print("TRACK in config.py:", config.TRACK, "" if config.TRACK else "<- one teammate sets this and pushes config.py")

The following code tells git who you are (name and email) so your commits are labelled "William Li" on GitHub. The noreply email keeps your real one private. Run it once per session, before committing

In [ ]:
!git config user.name "William Li"
!git config user.email "williamli218@users.noreply.github.com"

## 1.2 Download and load data

In [ ]:
from src import data

pool, test = data.load_track02()

## 1.3 Quick look

In [ ]:
data.quick_look(pool, "is_fraud", "train pool")
data.quick_look(test, "is_fraud", "test")
attacked = pool.groupby("cc_num", observed=True)["is_fraud"].max()
print(f"cards in pool: {len(attacked):,}, attacked (at least one fraud): {int(attacked.sum()):,}")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

counts = train["is_fraud"].value_counts().sort_index()
ax1.bar(["Legitimate", "Fraud"], counts.values, color=["#4C72B0", "#C44E52"])
ax1.set_yscale("log")
ax1.set_ylabel("Transactions (log scale)")
ax1.set_title(f"Class balance: {100 * train['is_fraud'].mean():.2f}% fraud")

hour = pd.to_datetime(train["trans_date_trans_time"]).dt.hour
by_hour = pd.crosstab(hour, train["is_fraud"], normalize="columns") * 100
ax2.plot(by_hour.index, by_hour[0], label="Legitimate")
ax2.plot(by_hour.index, by_hour[1], label="Fraud", color="#C44E52")
ax2.set_xlabel("Hour of day")
ax2.set_ylabel("% of that class")
ax2.set_title("When transactions happen")
ax2.legend()

plt.tight_layout()
plt.show()

## 1.4 Fixed test split

In [ ]:
print("train pool:", pool["trans_date_trans_time"].min(), "->", pool["trans_date_trans_time"].max())
print("test:      ", test["trans_date_trans_time"].min(), "->", test["trans_date_trans_time"].max())

## 1.5 Validation split
In order to choose for a validation cut date, a month-by-month view of transactions and fraud is displayed in the following table. `src/split.py`

In [ ]:
import pandas as pd
t = pd.to_datetime(pool["trans_date_trans_time"])
month = t.dt.to_period("M")

fraud = pool[pool["is_fraud"] == 1]
first_fraud = fraud.assign(t=t[fraud.index]).groupby("cc_num", observed=True)["t"].min()

monthly = pd.DataFrame({
    "rows": pool.groupby(month).size(),
    "fraud_tx": pool.groupby(month)["is_fraud"].sum(),
    "attacks_starting": first_fraud.dt.to_period("M").value_counts(),
}).fillna(0).astype(int)
monthly["fraud_rate_%"] = (100 * monthly["fraud_tx"] / monthly["rows"]).round(3)
monthly

fr = fraud.assign(t=t[fraud.index]).groupby("cc_num", observed=True)["t"].agg(["min", "max"])

rows = []
for cut in pd.date_range("2019-09-01", "2020-04-01", freq="MS"):
    rows.append({
        "cut": cut.date(),
        "train_months": round((cut - t.min()).days / 30.4, 1),
        "valid_months": round((t.max() - cut).days / 30.4, 1),
        "attacks_in_valid": int((fr["min"] >= cut).sum()),
        "straddling_cut": int(((fr["min"] < cut) & (fr["max"] >= cut)).sum()),
    })
pd.DataFrame(rows)

Choice of validation cut date: January 1st, 2020.

Done with helper functions from src/split.py

Reasons:
1. It gives a full year of training (all of 2019, every season, including holidays)
2. It gives a validation window close to the test set's length  (5.7 vs 6.3 months)
3. It gives plenty of attacks in validation (235, enough for a stable median)
4. Almost no attacks are split by the cut (only 2)

In [ ]:
from src import split

try:
    train, valid = split.make_validation_split(pool)
except NotImplementedError as e:
    train = valid = None
    print(e)
else:
    split.check_split(train, valid, *split.default_columns())
    print(f"train: {len(train):,} rows, valid: {len(valid):,} rows; both checks passed")

In [ ]:
!python -m pytest -q tests/test_leakage.py

## 1.6 Features
Helper function from `src/features.py`


In [ ]:
from src import features

features.build_features(pool.head(10))

## 1.7 The trivial baseline
The baseline calls every row legitimate: every score is 0. It's here to be scored with your metric and with accuracy, side by side, before you train anything. On this track it will look very accurate and catch nothing. That gap is the point of the evaluation metric section of your proposal. Helper function from src/metrics.py

In [ ]:
from src.models import always_legitimate
from sklearn.metrics import accuracy_score

baseline_scores = always_legitimate(len(pool))
print("baseline scores:", baseline_scores[:5], "... (all zeros)")
labelled = pool["is_fraud"] != -1
print("accuracy on the train pool if we predict 'legitimate' everywhere:",
      round(accuracy_score(pool["is_fraud"][labelled], (baseline_scores[labelled] >= 0.5).astype(int)), 4))

from src import metrics
y_valid = valid["is_fraud"].to_numpy()
s_valid = always_legitimate(len(valid))
print("headline metric:", metrics.median_frauds_before_hold(y_valid, s_valid, valid["cc_num"].to_numpy(), max_legit_hold_rate=0.01))
print("accuracy:       ", accuracy_score(y_valid, (s_valid >= 0.5).astype(int)))

## 1.8 Sequences
`make_sequences` turns the table into one fixed-length window per
transaction: the card's last `n` transactions ending at it, oldest first,
zero-padded while the card has fewer than `n`. `X` is what a sequence model
reads; `y` is the label of the last transaction in each window; `meta` keeps
the card, the time and how many rows in the window are real.

In [ ]:
X, y, meta = data.make_sequences(pool, n=10, feature_cols=["amt"])
print("X:", X.shape, X.dtype, "| y:", y.shape, "| meta columns:", list(meta.columns))
print("memory for X: %.0f MB" % (X.nbytes / 1e6))
i = int(meta.index[meta["n_valid"] == 10][0])
print("one window (amount only), oldest first:", X[i, :, 0].round(2))

In [ ]:
!python -m pytest -q tests/test_metrics.py tests/test_leakage.py

## 2.1 Simple baseline model: logistic regression
Train model and evaluate score `src/models.py`

In [ ]:
import numpy as np, pandas as pd
from src import features, models, metrics

feats = features.build_features(pool)
X_train, X_valid = feats.loc[train.index], feats.loc[valid.index]
model = models.train_logistic_regression(X_train, train["is_fraud"])
s_valid = model.predict_proba(X_valid)[:, 1]
print("headline metric:", metrics.median_frauds_before_hold(
    valid["is_fraud"], s_valid, valid["cc_num"], max_legit_hold_rate=0.01))

Added and tested smoothing window of different sizes

In [ ]:
v = pd.DataFrame({"card": valid["cc_num"].to_numpy(), "s": s_valid})
for k in [2, 3, 5]:
    s_k = v.groupby("card", observed=True)["s"].transform(lambda x: x.rolling(k, min_periods=1).mean()).to_numpy()
    print(k, metrics.median_frauds_before_hold(valid["is_fraud"], s_k, valid["cc_num"], max_legit_hold_rate=0.01))

Save the model in `models/`

In [ ]:
import json
scaler = model.named_steps["standardscaler"]
lr = model.named_steps["logisticregression"]

params = {
    "features": list(X_train.columns),
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist(),
    "coef": lr.coef_[0].tolist(),
    "intercept": float(lr.intercept_[0]),
    "smooth_k": 3,
}
with open("models/logistic_regression.json", "w") as f:
    json.dump(params, f, indent=2)

Log experiments in `experiments/log.csv`

In [ ]:
import csv, datetime

def log_run(model, settings, metric, value, notes=""):
    with open("experiments/log.csv", "a", newline="") as f:
        csv.writer(f).writerow([datetime.date.today().isoformat(), "William Li",
                                model, settings, "valid", metric, value, notes])

log_run("all_legitimate", "score 0 everywhere", "median_frauds_before_hold", 9.0, "D1.5 baseline")
log_run("all_legitimate", "score 0 everywhere", "accuracy", 0.9939)
log_run("logistic_regression", "improved features, class_weight=balanced", "median_frauds_before_hold", 9.0)
log_run("logistic_regression", "+ amount bands", "median_frauds_before_hold", 9.0, "AUC 0.989")
for k, val in [(2, 5.0), (3, 3.0), (5, 4.0)]:
    log_run("logistic_regression", f"+ amount bands + smoothing k={k}", "median_frauds_before_hold", val)

## 2.2 Main model: LSTM
`src/models.py`

In [ ]:
from src import models

d = models.prepare_lstm_data(pool, train, valid, n=10)
lstm, history = models.train_lstm(d["X_tr"], d["y_tr"], d["X_va"], d["y_va"])

In [ ]:
s_lstm = models.lstm_scores(lstm, d["X_va"])
cards_va = d["meta_va"]["cc_num"].to_numpy()
y_va = d["meta_va"]["is_fraud"].to_numpy()

med = metrics.median_frauds_before_hold(y_va, s_lstm, cards_va, max_legit_hold_rate=0.01)
med_smooth = metrics.median_frauds_before_hold(y_va, models.smooth_scores(s_lstm, cards_va), cards_va, max_legit_hold_rate=0.01)

print("LSTM:              ", med)
print("LSTM + smoothing 3:", med_smooth)

Save model in `models/` and data in `experiments/log.csv`

In [ ]:
torch.save(lstm.state_dict(), "models/lstm_n10.pt")
with open("models/lstm_n10.json", "w") as f:
    json.dump({"features": features.LSTM_COLS, "n": 10, "hidden_dim": 32,
               "mean": d["mean"].tolist(), "std": d["std"].tolist(), "history": history}, f, indent=2)

In [ ]:
log_run("lstm", "n=10, hidden=32, lr=1e-3, batch 2048, pos_weight, early stop (best epoch 11)", "median_frauds_before_hold", 2.0, "2.9 min on T4")
log_run("lstm", "same + smoothing k=3", "median_frauds_before_hold", 3.0)

## 2.3 Required experiments
Models comparison with median

In [ ]:
from src import metrics
import numpy as np

def M(y, s, cards):
    return metrics.median_frauds_before_hold(np.asarray(y), s, np.asarray(cards), max_legit_hold_rate=0.01)

p = models.load_logistic_regression()
s_lr_tr = models.logistic_regression_scores(p, feats.loc[train.index])
s_lr_va = models.logistic_regression_scores(p, feats.loc[valid.index])

s_lstm_tr = models.lstm_scores(lstm, d["X_tr"])

results = {
    "LR": (M(train["is_fraud"], s_lr_tr, train["cc_num"]),
           M(valid["is_fraud"], s_lr_va, valid["cc_num"])),
    "LR + smoothing (k=3)": (M(train["is_fraud"], models.smooth_scores(s_lr_tr, train["cc_num"]), train["cc_num"]),
                             M(valid["is_fraud"], models.smooth_scores(s_lr_va, valid["cc_num"]), valid["cc_num"])),
    "LSTM (n=10)": (M(d["meta_tr"]["is_fraud"], s_lstm_tr, d["meta_tr"]["cc_num"]),
                M(d["meta_va"]["is_fraud"], s_lstm, d["meta_va"]["cc_num"])),
}
for name, (tr, va) in results.items():
    print(f"{name:22s} train {tr:4.1f} | valid {va:4.1f}")

Figures for comparison

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

names = list(results.keys())
tr = [results[n][0] for n in names]
va = [results[n][1] for n in names]
x = np.arange(len(names))
ax1.bar(x - 0.2, tr, width=0.4, label="train")
ax1.bar(x + 0.2, va, width=0.4, label="validation")
ax1.axhline(9, color="grey", linestyle="--", label="all-legitimate baseline (9)")
ax1.set_xticks(x)
ax1.set_xticklabels(names)
ax1.set_ylabel("Median frauds before hold (lower is better)")
ax1.set_title("LSTM vs logistic regression, at 1% of legit cards held")
ax1.legend()

ep = [h["epoch"] for h in history]
ax2.plot(ep, [h["train_loss"] for h in history], marker="o", label="train loss")
ax2.plot(ep, [h["valid_loss"] for h in history], marker="o", label="validation loss")
best = min(history, key=lambda h: h["valid_loss"])["epoch"]
ax2.axvline(best, color="grey", linestyle="--", label=f"best epoch ({best}), kept by early stopping")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Weighted BCE loss")
ax2.set_title("LSTM training: overfitting starts after the best epoch")
ax2.legend()

plt.tight_layout()
plt.show()

## 2.4 Results table
Precision and recall calculations

In [ ]:
import pandas as pd

def operating_point(y, s, cards, max_rate=0.01):
    """Card-level precision and recall at the threshold the headline metric uses."""
    df = pd.DataFrame({"card": np.asarray(cards), "y": np.asarray(y).astype(int), "s": np.asarray(s, dtype=float)})
    per_card = df.groupby("card", sort=False).agg(attacked=("y", "max"), max_score=("s", "max"))
    legit = np.sort(per_card.loc[per_card["attacked"] == 0, "max_score"].to_numpy())
    cand = np.append(np.unique(df["s"]), np.inf)
    rate = (len(legit) - np.searchsorted(legit, cand, side="left")) / len(legit)
    t = cand[np.argmax(rate <= max_rate)]

    df["pos"] = df.groupby("card", sort=False).cumcount()
    first_hold = df[df["s"] >= t].groupby("card", sort=False)["pos"].min()
    last_fraud = df[df["y"] == 1].groupby("card", sort=False)["pos"].max()

    held = per_card.index.isin(first_hold.index)
    attacked = per_card["attacked"].to_numpy() == 1
    caught = last_fraud.index[first_hold.reindex(last_fraud.index) <= last_fraud]

    precision = (held & attacked).sum() / held.sum() if held.sum() else float("nan")
    recall = len(caught) / attacked.sum()
    return t, precision, recall

rows = {
    "All-legitimate": (valid["is_fraud"], np.zeros(len(valid)), valid["cc_num"]),
    "LR": (valid["is_fraud"], s_lr_va, valid["cc_num"]),
    "LR + smoothing (k=3)": (valid["is_fraud"], models.smooth_scores(s_lr_va, valid["cc_num"]), valid["cc_num"]),
    "LSTM (n=10)": (d["meta_va"]["is_fraud"], s_lstm, d["meta_va"]["cc_num"]),
}
for name, (y_, s_, c_) in rows.items():
    t, p, r = operating_point(y_, s_, c_)
    print(f"{name:22s} threshold {t:.4f} | precision {p:.2f} | recall {r:.2f}")

    log_run(name, "validation, 1% legit-card budget", "precision_cards", round(float(p), 3) if p == p else "nan")
    log_run(name, "validation, 1% legit-card budget", "recall_cards", round(float(r), 3))

## 2.5 Error analysis
Missed frauds and false holds

In [ ]:
import config

meta_va = d["meta_va"]
t_lstm, _, _ = operating_point(meta_va["is_fraud"], s_lstm, meta_va["cc_num"])

ev = meta_va[["cc_num", "is_fraud", "n_valid"]].copy()
ev["score"] = s_lstm
ev = ev.join(pool[["trans_date_trans_time", "amt", "category"]])
ev["gap_min"] = (np.expm1(feats.loc[ev.index, "log_seconds_since_prev"]) / 60).round(1)

g = ev.groupby("cc_num", observed=True, sort=False)
ev["flag"] = ev["score"] >= t_lstm
ev["held"] = g["flag"].cummax()
ev["fraud_no"] = g["is_fraud"].cumsum()
ev["card_attacked"] = g["is_fraud"].transform("max")

missed = ev[(ev["is_fraud"] == 1) & (~ev["held"])]
false_holds = ev[(ev["card_attacked"] == 0) & ev["flag"]].groupby("cc_num", observed=True).head(1)

print("threshold:", round(t_lstm, 4))
print("missed frauds:", len(missed), "| honest cards held:", len(false_holds))
print("\nwhich fraud in the burst got through (1 = first fraud on the card):")
print(missed["fraud_no"].value_counts().sort_index().head(10))

cols = ["trans_date_trans_time", "amt", "category", "gap_min", "n_valid", "fraud_no", "score"]

print("=== 5 random missed frauds ===")
print(missed.sample(5, random_state=config.SEED)[cols].to_string())

print("\n=== 5 random false holds (honest cards) ===")
print(false_holds.sample(5, random_state=config.SEED)[cols[:-2] + ["score"]].to_string())

## 2.6 Predict
Prepare `predict()` method in `src/predict.py` for test data and new incoming data

In [ ]:
!python -m pytest -q tests/test_predict.py

## 3.1 Threshold

In [ ]:
from src import decision

curve = decision.cost_curve(meta_va["is_fraud"], s_va, amt_va, meta_va["cc_num"])
best = decision.choose_thresholds(meta_va["is_fraud"], s_va, amt_va, meta_va["cc_num"])
print(best)
print(curve[curve["threshold"] >= 0.99917453].head(3).to_string())

## 3.2 Holding time

In [ ]:
import pandas as pd

t = pd.to_datetime(train["trans_date_trans_time"])
fraud = train[train["is_fraud"] == 1]
span = fraud.assign(t=t[fraud.index]).groupby("cc_num", observed=True)["t"].agg(["min", "max"])
hours = (span["max"] - span["min"]).dt.total_seconds() / 3600

print("attacks:", len(hours))
print("attack length in hours:")
print(hours.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1))

In [ ]:
times_va = pool.loc[meta_va.index, "trans_date_trans_time"]

for hours in [24, 48, 72, 96]:
    print(f"{hours:3d}h", decision.evaluate(s_va, meta_va["cc_num"], meta_va["is_fraud"],
                                            amt_va, times_va, best["hold"], hours))

In [ ]:
for name, hours in [("72h + re-hold", 72), ("hold forever", 10**7)]:
    print(f"{name:14s}", decision.evaluate(s_va, meta_va["cc_num"], meta_va["is_fraud"],
                                           amt_va, times_va, best["hold"], hours))

In [ ]:
import json
with open("models/thresholds.json", "w") as f:
    json.dump({"hold": best["hold"], "hold_hours": 72}, f, indent=2)

## 3.3 Test data

In [ ]:
import gc
import numpy as np, pandas as pd
from src import models, decision

for name in ["X", "X_va"]:
    if name in globals():
        del globals()[name]
gc.collect()

full = pd.concat([pool.assign(is_test=False), test.assign(is_test=True)], ignore_index=True)
full["cc_num"] = full["cc_num"].astype("int64")

X_f, _, meta_f = models.make_lstm_windows(full, n=m["n"])
X_f = models.scale_windows(X_f, meta_f, m["mean"], m["std"])

is_te = full.loc[meta_f.index, "is_test"].to_numpy()
X_te, meta_te = X_f[is_te], meta_f[is_te]
del X_f; gc.collect()

s_te = models.lstm_scores(m["lstm"], X_te)
amt_te = full.loc[meta_te.index, "amt"].to_numpy()
times_te = full.loc[meta_te.index, "trans_date_trans_time"]
times_va = pool.loc[meta_va.index, "trans_date_trans_time"]
print("test windows:", X_te.shape, "| fraud share %.4f" % meta_te["is_fraud"].mean())

D2_T = 0.9991745352745056
FOREVER = 24 * 365 * 10
rules = [("Doing nothing", np.inf, 72),
         ("LSTM, D2 threshold (hold forever)", D2_T, FOREVER),
         ("Decision layer (D3: cost threshold + 72h)", best["hold"], 72)]

def comparison(scores, meta_part, amts, times):
    return pd.DataFrame({name: decision.evaluate(scores, meta_part["cc_num"], meta_part["is_fraud"],
                                                 amts, times, t, h)
                         for name, t, h in rules}).T

print("\nTEST")
print(comparison(s_te, meta_te, amt_te, times_te).to_string())
print("\nVALIDATION (for comparison)")
print(comparison(s_va, meta_va, amt_va, times_va).to_string())

In [ ]:
import csv, datetime

FOREVER = 24 * 365 * 10
for name, t, h in [("all_legitimate", np.inf, 72),
                   ("lstm_d2_threshold", D2_T, FOREVER),
                   ("decision_layer_d3", best["hold"], 72)]:
    r = decision.evaluate(s_te, meta_te["cc_num"], meta_te["is_fraud"], amt_te, times_te, t, h)
    for metric in ["median frauds before hold", "honest cards held", "fraud charges through", "total cost $"]:
        with open("experiments/log.csv", "a", newline="") as f:
            csv.writer(f).writerow([datetime.date.today().isoformat(), "William Li", name,
                                    f"threshold {t}, hold {h}h", "test", metric, r[metric], "D3 test, run once"])